# Azure API inference

This notebook uses the same public runner as `main_api.py` and `run_API.sh`. Configuration, prompt truncation, few-shot tracking, result writing, traces, and reproducibility metadata therefore stay identical across CLI and notebook runs. Keep credentials in `configs/API.key.yaml` or environment variables; never paste keys into the notebook.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "main_api.py").is_file():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the clinical-llm-benchmark root")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.api.config import load_yaml, redact  # noqa: E402
from model.api_runner import run_api_from_config  # noqa: E402

print("PROJECT_ROOT:", PROJECT_ROOT)


## Inspect the run configuration

Edit the provider YAML rather than duplicating request parameters in notebook cells. `API.model.yaml` stores model capabilities; `API.key.yaml` is optional when credentials are provided through environment variables.

In [ ]:
config_path = PROJECT_ROOT / "configs" / "API.provider.azure.yaml"
model_config_path = PROJECT_ROOT / "configs" / "API.model.yaml"
key_path = PROJECT_ROOT / "configs" / "API.key.yaml"

run_config = load_yaml(config_path, required=True)
print(json.dumps(redact(run_config), indent=2, ensure_ascii=False))


## Validate paths without calling Azure

The dry run loads and validates the YAML/model registry and enumerates all result paths, but does not submit requests.

In [ ]:
dry_run = run_api_from_config(
    config_path=config_path,
    api_model_config_path=model_config_path,
    api_key_file=key_path,
    dry_run=True,
)
dry_run.result_paths


## Run inference

Set the guard to `True` only when the provider YAML points to the intended tasks, models, and output directory. Azure batch mode is selected from the same YAML/model registry used by the CLI.

In [ ]:
RUN_API = False

if RUN_API:
    run = run_api_from_config(
        config_path=config_path,
        api_model_config_path=model_config_path,
        api_key_file=key_path,
    )
    print("errors:", run.num_errors)
    print("results:", *run.result_paths, sep="\n- ")
else:
    print("Set RUN_API = True to submit the configured Azure run.")


## Reproduction artifacts

Each `*.result.json` is accompanied by `*.result.config.yaml` and `*.result.metadata.json`. When `save_trace` is enabled, exact requests and provider responses are also stored under `api_trace/<experiment>/`.